# Lab: Ridge, Lasso, and PCR
## CMSE 381 - Spring 2022
## March 18, 2022



In this module we are going to test out the ridge/lasso methods we discussed in class from Chapter 6.2, and the PCR ideas from Chapter 6.3.

In [ ]:
# Everyone's favorite standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import time


# ML imports we've used previously
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error



# Loading in the data

Ok, here we go, let's play with a baseball data set again. Note this cleanup is all the same as the last lab. 

In [ ]:
df = pd.read_csv('Hitters.csv').dropna().drop('Player', axis = 1)
df.info()
dummies = pd.get_dummies(df[['League', 'Division', 'NewLeague']])

In [ ]:
y = df.Salary

# Drop the column with the independent variable (Salary), and columns for which we created dummy variables
X_ = df.drop(['Salary', 'League', 'Division', 'NewLeague'], axis = 1).astype('float64')

# Define the feature set X.
X = pd.concat([X_, dummies[['League_N', 'Division_W', 'NewLeague_N']]], axis = 1)

X.info()

# Ridge Regression

In class, we learned that doing ridge regression means that we try to find the best model accoding to the score
$$
RSS + \lambda \sum_{i} \beta_i^2.
$$
The good news is that `scikitlearn` has a built in `Ridge` function.  

- [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)
- [User guide](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression)

In [ ]:
from sklearn.linear_model import Ridge

The bad (ok, not honestly that bad) news is that they call their $\lambda$ parameter $\alpha$. So we're just going to minimize 
$$
RSS + \alpha \sum_{i} \beta_i^2.
$$
instead. So if I pick an alpha value, I can do ridge regression as follows.

In [ ]:
ridge = Ridge(normalize = True) 

a = 1 #<------ this is me picking an alpha value

ridge.set_params(alpha = a)
ridge.fit(X, y)

print('intercept:', ridge.intercept_)
print('\n')
print(pd.Series(ridge.coef_, index = X.columns))
print('\nTraining MSE:',mean_squared_error(y,ridge.predict(X)))
    



&#9989; **<font color=red>Q:</font>** What is the `normalize=True` bit doing in the code above?

* Your answer here*

Of course, that was just me picking a random $\alpha$ out of a hat so there's no reason to trust that it's a good one. I could sit here all day and move that $\alpha$ around to see what's going on, but why do that, when I can make a for loop!

Here's a pile of $\alpha$s for us to test on.

In [ ]:
alphas = 10**np.linspace(4,-2,100)*0.5
alphas = np.append(alphas,0)
alphas

First off, let's take a look at how the coefficients learned change for various choices of $\alpha$. 

Associated with each alpha value is a vector of ridge regression coefficients, which we'll store in a matrix coefs. In this case, it is a  19×100  matrix, with 19 rows (one for each predictor) and 100 columns (one for each value of alpha). 

In [ ]:
ridge = Ridge(normalize = True)
coefs = []

for a in alphas:
    ridge.set_params(alpha = a)
    ridge.fit(X, y)
    coefs.append(ridge.coef_)
    
np.shape(coefs)

In [ ]:
ax = plt.gca()
ax.plot(alphas, coefs)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')


&#9989; **<font color=red>Q:</font>** There are two variables that have higher magnitude than the rest for low $\alpha$s (read: are either very large positive or very large negative). Which two are they from the data set? Which is which?


In [ ]:
# Your code here #

Now we can start setting up the usual train/test splits to have at least a starting idea of how the testing error is going. The `random_state=1` bit just makes it so that everyone should get the same random split. 

In [ ]:
# Split data into training and test sets
X_train, X_test , y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)


&#9989; **<font color=red>Do this:</font>** Train a model using ridge regression with $\alpha = 4$. What is the MSE of your model on the training set?

In [ ]:
# Your code here #


&#9989; **<font color=red>Do this:</font>** Ha ha nah, you can do better than that.  Lets try all our alphas and take a look at the testing MSE to make a better decision about what $\alpha$ we might want. Modify the code below to plot your testing MSE for all the alphas. What $\alpha$ should we use to train the model?

In [ ]:
# Modify your code from above and add it in the for loop to plot the testing MSE

ridge = Ridge(normalize = True)
errors = []

for a in alphas:
    # ==== Your code goes in here ==== #
    errors.append(17) #<----- random number in here so that the code runs before you fix it
    
np.shape(errors)

plt.plot(alphas,errors)
plt.title('Testing MSE')
ax=plt.gca()
ax.set_xscale('log')
ax.set_yscale('log')



## RidgeCV

Whelp, your meanie professor didn't tell you that there's actually a built in function to do this for you (sorry-not-sorry). Aren't you glad you didn't read ahead?


In [ ]:
from sklearn.linear_model import RidgeCV

- [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)
- [User Guide](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression)

Basically, `RidgeCV` runs LOOCV (unless you tell it otherwise, see the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)) on all the alpha values you specify on an input array, and tells you the best $\alpha$ given that. 

In [ ]:
# I'm going to drop that 0 from the alphas because it makes 
# RidgeCV cranky
alphas = alphas[:-1]


ridgecv = RidgeCV(alphas = alphas, 
                  scoring = 'neg_mean_squared_error', 
                  normalize = True)
ridgecv.fit(X_train, y_train)
print('alpha chosen is', ridgecv.alpha_)

I can predict my values on the test set directly from the `ridgecv` model we just built. 

In [ ]:
pred = ridgecv.predict(X_test)
mean_squared_error(y_test,pred)

This is exactly the same result as if I went and retrained my model using the chosen $\alpha$ using `Ridge`. 

In [ ]:
ridge = Ridge(alpha = ridgecv.alpha_, normalize = True)
ridge.fit(X_train, y_train)
mean_squared_error(y_test, ridge.predict(X_test))

&#9989; **<font color=red>Do this:</font>** Why did we get a different best choice of $\alpha$ than we found in the previous section? 

*Your answer here*

# Lasso 

Thanks to the wonders of `scikit-learn`, now that we know how to do all this with ridge regression, translation to lasso is super easy. 

- [Lasso Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
- [LassoCV Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html#sklearn.linear_model.LassoCV)
- [User guide](https://scikit-learn.org/stable/modules/linear_model.html#lasso)



In [ ]:
from sklearn.linear_model import Lasso, LassoCV

In [ ]:
# Here's a quick lasso code for a fixed $\alpha$
lasso = Lasso(max_iter = 10000, normalize = True)
# max_iter increases how long the lasso model tries to find a good solution
# In our case, if I leave it at the default 1000 I was getting errors, so I
# upped the value. 
lasso.set_params(alpha=1)
lasso.fit(X_train, y_train)

mean_squared_error(y_test,lasso.predict(X_test))


&#9989; **<font color=red>Do this:</font>** Make a few graphs similar to what we did in the previous section, but using Lasso instead of Ridge. 
- A graph of the coeffiencts as alpha changes 
- A graph of the test mean squared error as alpha changes

In [ ]:
# Your code for a graph of the coefficients goes here 

In [ ]:
# Your code for a graph of the MSEs goes here

&#9989; **<font color=red>Do this:</font>** Now try what we did with `LassoCV`.  What choice of $\alpha$ does it recommend? 

*I would actually recommend either not passing in any $\alpha$ list or passing explicitly `alphas = None`. `RidgeCV` can't do this, but `LassoCV` will automatically try to find good choices of $\alpha$ for you.*

In [ ]:
# Your code here

Now let's take a look at some of the coefficients. 

In [ ]:
# Some of the coefficients are now reduced to exactly zero.
pd.Series(lassocv.coef_, index=X.columns)

&#9989; **<font color=red>Q:</font>** We've been repeating over and over that lasso gives us coefficients that are actually 0.  At least in my code, I'm not seeing any that are 0. What happened?

*Your answer here*

In [ ]:
# You might also want some code in here to try to figure it out

# Principal Component Regression 

Ok, let's take a hard left turn and go try out some of the dimension reduction methods from Section 6.3. `Scikit-learn` doesn't have a built in function to do PCR (aka PCA and then regression) but it's just as easy for us to do it ourselves. 

First step, let's figure out the `PCA` function.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale #<--- this does the scaling of variables for us


In [ ]:
pca = PCA()
print(X.shape)
X_reduced = pca.fit_transform(scale(X))
print(X_reduced.shape)

"But Dr. Munch, you said PCA was supposed to do dimension reduction, why is my feature output the same size?"

Glad you asked, young data scientist. The PCA command outputs all of the PCs, all the way up through $p=19$ the original number of dimensions. 

So, if I want the first three PCs, I can get them out as follows. I'll put it in a data frame just to add column labels, but you don't need to do that.

In [ ]:
First3PCs = X_reduced[:,:3]

pd.DataFrame(First3PCs, columns = ['Z1','Z2', 'Z3'])

In [ ]:
pd.DataFrame(np.array(X)[:,:3], columns = ['X1','X2', 'X3'])

Now we can just do regression on the PCs like before. 

In [ ]:
from sklearn.linear_model import LinearRegression

regr = LinearRegression()
regr.fit(X_reduced[:,:3], y)
mean_squared_error(y,regr.predict(X_reduced[:,:3]))

&#9989; **<font color=red>Do this:</font>** My code above contains the rookie mistake of only reporting training error. Write modified code to return the 10-fold CV error of linear regression on the first 3 PCs.

In [ ]:
# Your code here #



# Hint: while not necessary, the following command might make 
# for some massively shortened code
from sklearn.model_selection import cross_val_score


&#9989; **<font color=red>Do this:</font>** Take the code you figured out above to get the score for 10-fold CV on the first 3 PCs, and include it in the for-loop below to see how the MSE changes as the number of PCs you use changes. 

In [ ]:
n = len(X_reduced)
regr = LinearRegression()
mse = []

# Calculate MSE using CV for the 19 principle components, adding one component at a time.
for i in np.arange(1, 20): # i is the number of principal components to use each time
    # ====
    score = 0 # Your code from above goes here!
    # ====

    mse.append(score)
    
# Plot results    
plt.plot(mse, '-v')
plt.xlabel('Number of principal components in regression')
plt.ylabel('MSE')
plt.title('Predicting Salary')
plt.xlim(xmin=-1);

&#9989; **<font color=red>Q:</font>** Based on the graph you generated above, how many PCs do you think you should use? 

*Note: Based on graphs I generated, I can see a few different options for what I might decide to use for number of principal components. This is one of those cases where you potentially have a different answer and/or reasoning from your neighbor, so I enourage you to talk this one through with your group.*


*Your answer here*

In [ ]:
##ANSWER

# Leaving this code down here since might want it later, but we didn't discuss this 
# in class so thinking we just drop it.
varExplained = np.cumsum(np.round(pca.explained_variance_ratio_, decimals=4)*100)

plt.plot(np.arange(1, 20),varExplained, '-v')

&#9989; **<font color=red>Q:</font>** Of the models you've built so far (Ridge, Lasso, and PCR), which would you choose to use and why? 

*Note: This goes in the no-one-right-answer bucket. Go argue with your group.*


*Your answer here*

# Principal Least Squares (PLS)

Just for completeness, here's code to do PLS. `Scikit-learn` `PLSRegression` gives the same results as the `pls` package in R when using `method='oscorespls'`. However, the standard method used in `sklearn` is `'kernelpls'`, which we'll use here. 

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold

In [ ]:
n = len(X_train)

# 10-fold CV, with shuffle
kf_10 = KFold(n_splits=10, shuffle=True, random_state=48864)

mse = []

for i in np.arange(1, 20):
    pls = PLSRegression(n_components=i)
    score = cross_val_score(pls, X_train, y_train, cv=kf_10, scoring='neg_mean_squared_error').mean()
    mse.append(-score)

# Plot results
plt.plot(np.arange(1, 20), np.array(mse), '-v')
plt.xlabel('Number of components in regression')
plt.ylabel('MSE')
plt.title('Salary')
plt.xlim(xmin=-1)

&#9989; **<font color=red>Q:</font>** After seeing the results of PLS, would you change your previous answer to the choice of model to use? Why or why not?

*Your answer here*


# Lab Survey

To get credit for today's lab, fill out the following survey before the end of class:

https://forms.gle/hX8GT5FJ2fNMeTo1A

Note this is the same link for every lab, so you will fill this out multiple times this semester.



-----
### Congratulations, we're done!
Written by Dr. Liz Munch, Michigan State University

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.